## Tutorial 0 - PyTorch as a Mathematical Optimization Framework

### Tensor creation & basic ops
How PyTorch is capable of potentially equivalent to NumPy with additional functionalities

In [ ]:
import torch
import numpy as np

np_x = np.array([[1., 2.], [3., 4.]])
torch_x = torch.tensor([[1., 2.], [3., 4.]])

print(np_x + 1)
print(torch_x + 1)

print(np_x @ np_x)
print(torch_x @ torch_x)

### Dtypes & devices
How can we assign datatypes implicitly with variables and what is the notion of devices in PyTorch

In [13]:
a = torch.randn(3, dtype=torch.float32)
b = torch.randn(3, dtype=torch.float64)

print(a.dtype, b.dtype)
print(a, b)

torch.float32 torch.float64
tensor([-0.4248,  0.1543, -0.0207]) tensor([-0.0992,  0.2312, -0.2109], dtype=torch.float64)


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.randn(3, 3).to(device)
y = torch.randn(3, 3).to(device)

print(x.device, y.device)
print(x + y)

cpu cpu
tensor([[-0.0992,  0.0675,  1.6396],
        [-0.6260,  1.3622,  2.0534],
        [ 1.7945, -1.1275, -1.3535]])


In [ ]:
device1 = torch.device("cuda")
device2 = torch.device("cpu")

x = torch.randn(3, 3).to(device1)
y = torch.randn(3, 3).to(device2)

print(x.device, y.device)
print(x + y)

#### Why is such distribution of multiple devices needed? Can't we use a single device to achieve this?
- So, typically while working with Deep learning models, we can exploit this functionality of parrallely processing massive datasets, while allowing simple processing on sequential tasks. 
- Training neural networks requires billions of matrix multiplications, which the GPU handles in parallel, while the CPU handles loading and shuffling data.

### Autograd: How is PyTorch Working
The goal is to understand exactly how PyTorch tracks operations and computes gradients

#### Scalars without autograd: No gradient tracking

In [ ]:
x = torch.tensor(2.0)
y = x**2 + 3*x + 1

y.backward()

This throws an error. PyTorch does not track gradients by default. You must explicitly opt in.

#### Scalars with autograd enabled: Enable gradient tracking

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x + 1

y.backward()
x.grad

- x.grad stores ∂y/∂x
- backward() basically means: *Differentiate y w.r.t. all leaf tensors that require grad*

#### Inspecting the computation graph: grad_fn

In [ ]:
print(y.grad_fn)

In [ ]:
y.grad_fn.next_functions

The key idea behind all these operations is the underlying principle of how PyTorch dynamically handles data:
- Each operation (+, *, **) creates a node
- PyTorch builds a **DAG** (directed acyclic graph) dynamically

So to speak, PyTorch uses define-by-run, not a static graph.

#### Leaf tensors vs intermediate tensors

In [19]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
z = y ** 2

z.backward()

In [ ]:
print(x.grad)
print(y.grad)

Gradients are stored only for leaf tensors (user-created parameters)

#### Gradient accumulation: Call backward twice

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2

y.backward()
print(x.grad)

y.backward()
print(x.grad)

PyTorch uses a dynamic computational graph. By default, the intermediate buffers used to calculate gradients are released after the first call to backward() to optimize memory usage. The second call attempts to traverse a graph that no longer exists, resulting in the runtime error. 
[[Article](https://github.com/mrdbourke/pytorch-deep-learning/discussions/334#:~:text=the%20training%20loop:-,RuntimeError:%20Trying%20to%20backward%20through%20the%20graph%20a%20second%20time,saved%20tensors%20after%20calling%20backward.)]

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2

y.backward(retain_graph=True)
print(x.grad)

y.backward()
print(x.grad)

PyTorch adds gradients by default. Formally:
$$grad ← grad + \frac{∂𝑦}{∂𝑥}$$
This enables multiple loss terms, and also gradient accumulation across mini-batches

You can resolve the lack of graph issue in two main ways:
1. Retain the graph for multiple uses (as seen above): Pass *`retain_graph=True`* in the first (and all but the last) call to backward() to prevent the intermediate buffers from being freed. **Note that this consumes more memory**.
2. Create a new graph for each backward pass (typical for training loops): In a standard training loop, you create a new computation graph in every iteration during the forward pass. You also need to explicitly clear old gradients using *`optimizer.zero_grad()`* before the next backward pass to prevent them from accumulating.

#### Clearing gradients properly

In [ ]:
x.grad.zero_()
print(x.grad)

The important point to note is that since the gradients accumulate everytime `backward()` function is called, it is very important to **zero gradients before the next backward pass**.

#### Gradient Descent Explanation
- For now, lets focus on understanding how the maths translates into action with everything we have seen so far.
- So, Training = Forward pass $\rightarrow$ Backward pass $\rightarrow$ Parameter update. No neural networks yet. Just math.

#### Loss landscape intuition

We minimize:
$$𝑓(𝑥) = (𝑥 - 3)^2$$
Two questions we would like to answer are:
1. Where is the minimum?
2. What does the gradient tell us?

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
lr = 0.1

for step in range(3):
    loss = (x - 3)**2
    loss.backward()
    x = x - lr * x.grad

##### What is wrong here?
1. **Breaking the "Chain"**: When you do `x = x - ...`, you create a new tensor that no longer has `requires_grad=True`. In the second loop, PyTorch will see that x is just a plain number and will throw an error saying: *RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn*
2. **The "In-Place" Modification Rule**: PyTorch tracks every operation on x to build a history. If you modify x while PyTorch is watching, it gets confused. To update x without it being part of the gradient calculation, you must wrap the update in with `torch.no_grad()`
3. **Gradient Accumulation (The "8" problem from before)**: As we discussed, PyTorch adds gradients. If you don't clear x.grad at the end of each loop, the second step will use a massive, incorrect gradient.

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
lr = 0.1

for step in range(10):
    loss = (x - 3)**2
    loss.backward()
    
    with torch.no_grad():
        x -= lr * x.grad
    
    x.grad.zero_()
    
    print(f"Step {step}: x={x.item():.4f}, loss={loss.item():.4f}")

#### Optional mini-exercise
You can also further experiment by trying the following exercises:
1. Try changing learning rates
2. Try to look at examples for convergence vs divergence
3. Also try the same exercise for (x-3)**4

### Summary of tensors

torch.Tensor is the central class of the package. If you set its attribute .requires_grad as True, it starts to track all operations on it. When you finish your computation you can call .backward() and have all the gradients computed automatically. The gradient for this tensor will be accumulated into .grad attribute.

To stop a tensor from tracking history, you can call .detach() to detach it from the computation history, and to prevent future computation from being tracked.

To prevent tracking history (and using memory), you can also wrap the code block in with torch.no_grad():. This can be particularly helpful when evaluating a model because the model may have trainable parameters with requires_grad=True, but we don't need the gradients.

There’s one more class which is very important for autograd implementation - a Function.

Tensor and Function are interconnected and build up an acyclic graph, that encodes a complete history of computation. Each variable has a .grad_fn attribute that references a Function that has created the Tensor (except for Tensors created by the user - their grad_fn is None).

If you want to compute the derivatives, you can call .backward() on a Tensor. If Tensor is a scalar (i.e. it holds a one element data), you don’t need to specify any arguments to backward(), however if it has more elements, you need to specify a gradient argument that is a tensor of matching shape.